# Lab 4A: Open-Loop Step Response

Run the cells in order. The MicroPython program remains active and waits for
commands after the serial connection is established.

In [ ]:
from serial import Serial
from serial.tools import list_ports
import time
import numpy as np
from matplotlib import pyplot as plt

## 1. Open the Serial Connection

List the available serial ports and identify that same Shoe USB port. On Windows,
it will usually look like `COM3`. On macOS, use the `/dev/cu.usbmodem...` port.

In [ ]:
for port in list_ports.comports():
    print(port.device, port.description)

Set `SERIAL_PORT` to the port found above. This is the only value that
you need to change in the serial-connection setup.

In [ ]:
SERIAL_PORT = "COM3"  # Change this value.
BAUDRATE = 115200

Open the serial port, restart the MicroPython program, and wait until
`main.py` prints `READY LAB4A_SERIAL_V1`.



In [ ]:
ser = Serial(SERIAL_PORT, baudrate=BAUDRATE, timeout=0.2)
time.sleep(0.3)
ser.reset_input_buffer()

# Ctrl-B returns to the normal REPL, Ctrl-C stops a running program,
# and Ctrl-D soft-resets MicroPython so main.py runs again.
ser.write(b"\x02")
time.sleep(0.1)
ser.write(b"\x03")
time.sleep(0.1)
ser.write(b"\x04")

deadline = time.time() + 5
ready = False

while time.time() < deadline and ready == False:
    line = ser.readline().decode().strip()

    if line == "READY LAB4A_SERIAL_V1":
        ready = True
        print(line)

if ready == False:
    print("ERR did not receive READY. Check that main.py is uploaded and has no errors.")
else:
    print("Serial port is open and main.py is running.")

## 2. Define `run_command()`

`run_command()` sends one command through the open serial connection and reads
the response through `END`. After responding, the MicroPython program
immediately waits for the next command.

**Provided code — do not modify this cell.**

In [ ]:
def run_command(command, timeout=10):
    lines = []

    # Send one complete command line to the board.
    ser.write((command + "\r\n").encode())

    deadline = time.time() + timeout
    finished = False

    while time.time() < deadline and finished == False:
        current_line = ser.readline().decode().strip()

        if current_line != "":
            lines.append(current_line)

            if current_line == "END":
                finished = True
            elif current_line.startswith("DATA,") == False:
                print(current_line)

    if finished == False:
        print("ERR no END before timeout")

    return lines

## 3. Run the Step-Response Test

Set `STEP_PERCENT` to the step-input percentage specified in the lab manual.
The provided code adds this value to the `RUN_STEP` command. Run the cell whenever
you want the microcontroller to perform one step-response test. The returned lines
are stored in `response_lines` for the next cell.

In [ ]:
# TODO: Replace 0 with the required step-input percentage.
STEP_PERCENT = 0

# Provided command code — do not modify.
response_lines = run_command(
    f"RUN_STEP {STEP_PERCENT}",
    timeout=10,
)

## 4. Save the Data

Extract the `DATA` lines from the most recent command response and save the
time and velocity columns in a CSV file.

**Provided code — do not modify this cell.**

In [ ]:
DATA_FILENAME = "data.csv"

data_lines = [
    line[len("DATA,"):]
    for line in response_lines
    if line.startswith("DATA,")
]

if len(data_lines) == 0:
    print("ERR no data were returned by the microcontroller")
else:
    with open(DATA_FILENAME, "w") as data_file:
        for data_line in data_lines:
            data_file.write(data_line + "\n")

    print(f"Saved {len(data_lines)} samples to {DATA_FILENAME}.")

## 5. Load and Plot the Data

Load the CSV data into a NumPy array, then separate its two columns into time
and velocity arrays.

In [ ]:
data = np.genfromtxt(DATA_FILENAME, delimiter=",")
times = data[:, 0]
velocities = data[:, 1]

Plot the step response. Label both axes with units and give the plot a
descriptive title. Matplotlib commands are similar to MATLAB commands but use the
`plt.` prefix.

In [ ]:
# TODO: Plot the measured velocity versus time.

## 6. Close the Serial Connection

After all tests are complete, close the serial port so that Thonny or another
program can use it. Rerun the connection cell before sending
additional commands.

In [ ]:
ser.close()